# 06c tokenizer ablation: amount bins and the time-gap token

One of the two groups of step 6c (the other is `06c_objective.ipynb`). Each arm pretrains its own model from scratch on this machine with its own tokenizer, then fine-tunes it for fraud exactly as step 4 did (H = 12, last-token pooling, three seeds). The control arm A is retrained here too, so every comparison in this notebook is between runs of one machine (same-machine rule, see 06b). The two groups write disjoint folders (`ablate_pre/tokenizer/`) and disjoint result files (`06c_tokenizer_...`), so they can run on two machines at the same time.

| arm | tokenizer | objective | talk it tests |
|---|---|---|---|
| A_ref | step 2's: amount in 20 log bins, time-gap token on | CLM | the control |
| B_linear | amount in 20 linear bins | CLM | WF: "data shape matters more than data size" |
| C_no_dt | no time-gap token (6 tokens per transaction) | CLM | same; step 5 found the rhythm features unimportant on Sparkov |

**Budget.** Three pretraining runs (8 epochs, about 8,000 steps: about 37 min each on a T4, about 1.5 h on Apple MPS) plus nine fine-tuning runs (about 10 min on a T4, about 23 min on MPS). Everything is resumable: a disconnect loses at most one save interval, a finished arm or run is skipped by its json. Rerun the notebook from the top after a disconnect.

In [ ]:
import os, sys, glob, shutil
try:
    from google.colab import drive, userdata                      # Colab: mount Drive, pull the repo, sync new notebooks to Drive
    drive.mount("/content/drive")
    tok = userdata.get("GH_TOKEN")
    if not os.path.exists("/content/transaction-fm"):
        !git clone -q https://{tok}@github.com/lb603487485/transaction-fm.git /content/transaction-fm
    else:
        !cd /content/transaction-fm && git pull -q
    %pip install -q datasets xgboost
    %cd /content/transaction-fm
except ImportError:                                               # laptop: this notebook is the repo copy; data under $TFM_ROOT
    here = os.getcwd()
    while not os.path.isdir(os.path.join(here, "src", "tfm")):
        assert here != os.path.dirname(here), "run this notebook from inside the transaction-fm repo"
        here = os.path.dirname(here)
    %cd {here}
sys.path.insert(0, os.path.abspath("src"))
from tfm.runtime import root, describe, on_colab
ROOT = root()
os.makedirs(f"{ROOT}/notebooks", exist_ok=True)
if on_colab():
    for f in glob.glob("notebooks/*.ipynb"):                      # new notebooks from the repo appear in Drive
        dst = f"{ROOT}/notebooks/{os.path.basename(f)}"
        if not os.path.exists(dst):
            shutil.copy(f, dst); print("new notebook in Drive:", os.path.basename(f))
import tfm
print("tfm ok |", describe(), "| ROOT =", ROOT)

## Stage 6c-0: the arms' tokenizers, side by side

`arm_data` fits each arm's tokenizer on train rows (through `assert_train_only`), encodes every row with it and cuts the 512-token windows for train and val, cached under the arm folder. Arms that share a tokenizer configuration still fit and store their own copy, so an arm folder is self-contained. The example row shows what changed between arms. The three dicts at the top stay empty in a real run; the smoke run on the synthetic fixture fills them to make the models tiny and the runs short.

In [ ]:
import json, time, os, numpy as np, pandas as pd
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)
from pathlib import Path
from tfm.finetune import build_examples
from tfm.metrics import mean_sd
from tfm.ablate import finetune_run, run_name
from tfm.ablate_pretrain import ARMS, GROUPS, SEEDS, arm_tokenizer_config, arm_data, pretrain_arm, finetune_spec, pretrained_dir
from tfm.runtime import accelerator

GROUP = "tokenizer"
ARM_LIST, SNAP_ARM = GROUPS[GROUP]["arms"], GROUPS[GROUP]["snapshots"]
EPOCHS = 8                                   # the guide's pretraining length, as in step 3
PRE_OVERRIDES, FT_OVERRIDES, CFG = {}, {}, {}   # empty in a real run; the smoke run sets tiny values here
RES = f"{ROOT}/results"; os.makedirs(RES, exist_ok=True)
df = pd.read_parquet(f"{ROOT}/data/processed/transactions.parquet")
card, split, fraud = df["card_id"].to_numpy(), df["split"].to_numpy(), df["is_fraud"].to_numpy()
DEV = accelerator()
print("group:", GROUP, "arms:", ARM_LIST, "| accelerator:", DEV, describe())

toks, windows, rows = {}, {}, []
for arm in ARM_LIST:
    tok, tw, vw = arm_data(GROUP, arm, df, ROOT)
    toks[arm], windows[arm] = tok, (tw, vw)
    rows.append({"arm": arm, "objective": ARMS[arm]["objective"], **{k: v for k, v in arm_tokenizer_config(arm).items() if k in ("amt_binning", "use_dt")},
                 "vocab": len(tok.vocab), "tokens/tx": tok.tokens_per_tx, "train windows": len(tw), "val windows": len(vw),
                 "example row": " ".join(tok.decode(tw[0][:tok.tokens_per_tx]))})
display(pd.DataFrame(rows))

## Stage 6c-1: pretrain the arms

Same settings as step 3 (lr 5e-4, cosine, 5 percent warmup, batch 32, eval and save every 500 steps, best by validation loss). The snapshot arm, if this group has one, additionally saves weight-only snapshots at log-spaced steps. An MLM arm's loss is over masked positions only, so its perplexity is not comparable with a CLM arm's and is reported but not plotted; the per-field accuracy is the comparable number (next-token for CLM, masked-token for MLM, both against the same train prior).

In [ ]:
t_all = time.time()
pre = {}
for arm in ARM_LIST:
    pre[arm] = pretrain_arm(GROUP, arm, df, ROOT, RES, epochs=EPOCHS, overrides=PRE_OVERRIDES, cfg_overrides=CFG)
print(f"\npretraining stage: {(time.time() - t_all) / 60:.1f} min this session")
ptab = pd.DataFrame([{"arm": a, "objective": s["objective"], "vocab": s["vocab_size"], "params M": round(s["n_params"] / 1e6, 2), "train windows": s["n_train_windows"],
                      "best step": s["best_step"], "val loss": s["best_eval_loss"], "perplexity": s["best_perplexity"],
                      **{f"acc {f} - prior": s["per_field_accuracy_at_best"].get(f, float("nan")) - s["prior_accuracy"].get(f, float("nan")) for f in ("DT", "AMT", "MER", "CAT")},
                      "snapshots": s["snapshots"], "min": s["wall_min"], "where": s["runtime"].get("accelerator", s["runtime"]["device"])} for a, s in pre.items()])
display(ptab.style.format({"val loss": "{:.4f}", "perplexity": "{:.2f}", **{c: "{:+.3f}" for c in ptab.columns if c.startswith("acc ")}}).hide(axis="index"))

## Stage 6c-2: fine-tune every arm (3 seeds), and the snapshots (seed 0) if this group has them

`finetune_run` is the step-4 loop as a function (06b). Each arm encodes the data with its own tokenizer, so the examples are built per arm; an MLM arm is loaded as a BERT body with a last-token head, the same pooling as the decoder arms, so the comparison changes the objective and nothing else.

In [ ]:
t_all = time.time()
ids_all, examples, ft = {}, {}, []
for arm in ARM_LIST:
    tok = toks[arm]; ids_all[arm] = tok.encode(df); examples[arm] = {}
    PAD, BOS, V = tok.vocab["<PAD>"], tok.vocab["<BOS>"], len(tok.vocab)
    specs = [finetune_spec(GROUP, arm, s) for s in SEEDS] + ([finetune_spec(GROUP, arm, 0, snapshot=st) for st in pre[arm]["snapshots"]] if arm == SNAP_ARM else [])
    for r in specs:
        ft.append(finetune_run(r, ids_all[arm], card, split, fraud, PAD, BOS, V, pretrained_dir(r, ROOT), ROOT, RES, examples[arm], overrides=FT_OVERRIDES))
print(f"\nfine-tuning stage: {(time.time() - t_all) / 60:.1f} min this session, {len(ft)} runs")
KEYS = ["pr_auc", "roc_auc", "recall_at_alert", "card_discovery", "fp_ratio"]
ftab = pd.DataFrame([{"run": s["name"], "arm": s["arm"], "snapshot": s["snapshot"], "seed": s["seed"], "best step": s["best_step"], "val PR-AUC": s["val_at_best"]["pr_auc"],
                      **{k: s["test"][k] for k in KEYS}, "min": s.get("wall_min"), "where": s["runtime"].get("accelerator", s["runtime"]["device"])} for s in ft])
display(ftab.style.format({c: "{:.4f}" for c in ["val PR-AUC"] + KEYS}).hide(axis="index"))

## Stage 6c-3: the arms against A, and the scatter if this group has snapshots

Test metrics per arm, mean and sd over the three seeds, and the difference from A in units of A's seed sd. The scatter: for each snapshot, perplexity at that step (from the pretraining eval curve) against the test PR-AUC after fine-tuning, plus the final model (3-seed mean); Spearman's rho says how monotone the relation is over these points.

In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
full = [s for s in ft if s["snapshot"] is None]
arms = {a: mean_sd([s["test"] for s in full if s["arm"] == a]) for a in ARM_LIST if any(s["arm"] == a for s in full)}
ref = arms.get("A_ref")
atab = pd.DataFrame([{"arm": a, "objective": ARMS[a]["objective"], "n seeds": m["pr_auc"]["n_seeds"],
                      **{k: f"{m[k]['mean']:.4f} +- {m[k]['sd']:.4f}" for k in KEYS},
                      "PR-AUC - A": (m["pr_auc"]["mean"] - ref["pr_auc"]["mean"]) if ref else float("nan"),
                      "in A sd": ((m["pr_auc"]["mean"] - ref["pr_auc"]["mean"]) / max(ref["pr_auc"]["sd"], 1e-9)) if ref else float("nan")} for a, m in arms.items()])
display(atab.style.format({"PR-AUC - A": "{:+.4f}", "in A sd": "{:+.1f}"}).hide(axis="index"))

scatter, rho = pd.DataFrame(columns=["step", "perplexity", "pr_auc", "kind"]), float("nan")
if SNAP_ARM and SNAP_ARM in pre:
    curve = {int(st): float(l) for st, l in pre[SNAP_ARM]["eval_curve"]}
    pts = [{"step": s["snapshot"], "perplexity": float(np.exp(curve[s["snapshot"]])), "pr_auc": s["test"]["pr_auc"], "kind": "snapshot"}
           for s in ft if s["arm"] == SNAP_ARM and s["snapshot"] is not None and s["snapshot"] in curve]
    if SNAP_ARM in arms:
        pts.append({"step": pre[SNAP_ARM]["best_step"], "perplexity": pre[SNAP_ARM]["best_perplexity"], "pr_auc": arms[SNAP_ARM]["pr_auc"]["mean"], "kind": "best, 3-seed mean"})
    scatter = pd.DataFrame(pts, columns=["step", "perplexity", "pr_auc", "kind"]).sort_values("step")
    display(scatter)
    rho = float(spearmanr(scatter["perplexity"], scatter["pr_auc"]).correlation) if len(scatter) > 2 else float("nan")
    fig, ax = plt.subplots(figsize=(6, 4.2))
    for kind, g in scatter.groupby("kind"):
        ax.scatter(g["perplexity"], g["pr_auc"], marker="o" if kind == "snapshot" else "*", s=60 if kind == "snapshot" else 160, label=kind)
    for _, p in scatter.iterrows():
        ax.annotate(f"step {p['step']}", (p["perplexity"], p["pr_auc"]), textcoords="offset points", xytext=(5, 4), fontsize=8)
    ax.set_xlabel("validation perplexity at the pretraining step"); ax.set_ylabel("test PR-AUC after fine-tuning (H12, seed 0)")
    ax.set_title(f"intrinsic vs extrinsic, arm {SNAP_ARM}, {DEV}   Spearman rho = {rho:.2f}"); ax.invert_xaxis(); ax.legend()
    plt.tight_layout(); fig.savefig(f"{RES}/06c_{GROUP}_perplexity_vs_prauc.png", dpi=120); plt.show()
    print(f"Spearman rho(perplexity, PR-AUC) over {len(scatter)} points: {rho:.2f}  (-1 would mean lower perplexity always meant higher PR-AUC)")

## Save 06c_tokenizer.json

In [ ]:
out = {"group": GROUP, "pretrain": pre, "finetune_runs": ft, "arms": arms, "scatter": scatter.to_dict(orient="records"), "spearman_rho": rho, "epochs": EPOCHS, "runtime": describe()}
with open(f"{RES}/06c_{GROUP}.json", "w") as f:
    json.dump(out, f, indent=2, default=float)
print(f"saved 06c_{GROUP}.json:", list(out))

## What to read off the numbers

- Arms against A, in A's seed sd: a difference under about 2 sd is a tie at three seeds; say so. Tokenizer group: WF's "data shape" claim predicts that linear amount bins (B) hurt, since most rows land in one or two bins (02's figure); C (no time gap) may tie, as step 5 found the rhythm features unimportant on Sparkov. Objective group: perplexities are not comparable (D's is over masked positions); the per-field accuracy gains and the fine-tuned PR-AUC are. If D is close, the talks' disagreement does not matter at this scale; if D is clearly behind, WF's decoder choice held here.
- Scatter (objective group): the early snapshots will have high perplexity and low PR-AUC, so some correlation is guaranteed; the question is the late part. If the last two or three snapshots sit at nearly the same PR-AUC while perplexity still falls, that is Wealthsimple's point: intrinsic metrics stop predicting downstream value before training stops improving them.
- Every run json carries `runtime`; the report's tables state which machine produced each number. A appears in both groups, trained on whichever machine ran that group; the two A results are a second cross-machine data point next to 06b's G3, not a number to average.

## Push results and this notebook back to GitHub

In [ ]:
import shutil, glob, os, json
os.makedirs("results/metrics", exist_ok=True); os.makedirs("results/figures", exist_ok=True)
for f in glob.glob(f"{ROOT}/results/*.json"):
    shutil.copy(f, "results/metrics/")
for f in glob.glob(f"{ROOT}/results/*.png"):
    shutil.copy(f, "results/figures/")
if on_colab():
    def same_code(a, b):   # a Drive notebook may be copied back only if its code cells equal the repo version
        cells = lambda p: ["".join(c["source"]) for c in json.load(open(p))["cells"] if c["cell_type"] == "code"]
        return cells(a) == cells(b)
    for f in glob.glob(f"{ROOT}/notebooks/*.ipynb"):   # outputs are kept so GitHub shows them
        repo = f"notebooks/{os.path.basename(f)}"
        if not os.path.exists(repo) or same_code(f, repo):
            shutil.copy(f, repo)
        else:
            print("STALE, not copied:", os.path.basename(f), "-> delete it from Drive and rerun the setup cell")
    !git config user.email "linboyy@hotmail.com" && git config user.name "Bo Lin"
    !git add results notebooks && git commit -q -m "results and notebooks from Colab" && git pull --rebase -q && git push -q   # the other 06c group may have pushed meanwhile; its files are disjoint
    !git log --oneline | head -1
else:
    print("laptop: results copied into results/; this notebook file already is the repo copy; commit from the terminal")
    !git status --short results notebooks

In [ ]:
if on_colab():
    drive.flush_and_unmount()